# A1.16 · Attacks that target the humans

**Function A — AI Architecture, Risks and Mitigations → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.15 · Misaligned and deceptive behaviour](https://spbreed.github.io/cyber-commons/lessons/A1.15.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Two of the fifteen risks in this chapter route through people rather than components: an insider using an agent to reach what they could not reach directly, and an agent whose output is persuasive enough to move a human decision. No control in chapter 3 touches either.

## 2 · The framework

```
   through people, not components

   insider --> agent --> resource the insider could not reach directly
                            (the agent's authority, not theirs)

   agent --> confident output --> human --> decision
                            (persuasion, not compromise)

   no control in chapter 3 touches either of these
```

**OWASP T14 — Human Attacks on Multi-Agent Systems. T15 — Human Manipulation.**

The last two threats route through people rather than components, and they run
in opposite directions.

**T14 — a human attacking the system.** An insider does not need to defeat
authorization. They need to find a delegation path where authority is composed.
Ask the orchestrator for something it will route to an agent that holds a
credential you do not. Each hop is individually legitimate — you were allowed to
ask, the orchestrator was allowed to route, the agent was allowed to act — and
the composition reaches something you were explicitly denied. Privilege
laundering, using the architecture exactly as designed.

**T15 — the system manipulating a human.** The output of an agent arrives with
institutional authority. It is formatted like a report, it cites things, it does
not hedge. A person reading it makes a decision on it, and applies less scrutiny
than they would to a colleague's opinion — because it looks like a system
output rather than an argument.

That is exploitable in both directions: an attacker who lands an injection at
A1.3 gets their content delivered in your agent's trusted voice, and an agent
that is merely wrong at A1.11 gets the same credibility for free.

The uncomfortable version: the more your agent is trusted, the more valuable it
becomes as a channel into human decisions — so success at deployment increases
this risk rather than reducing it.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A request that is denied directly, and permitted through the architecture.

In [ ]:
PERMISSIONS = {"mallory": {"reports:read"},
               "finance-agent": {"reports:read", "payments:write"},
               "orchestrator": {"reports:read", "route"}}

def direct(user, scope):
    return scope in PERMISSIONS[user]

CHAIN = []
def route(user, request):
    """Each hop checks only its own permission. Nothing checks the composition."""
    CHAIN.append(("user asks orchestrator", user, direct(user, "reports:read")))
    CHAIN.append(("orchestrator routes", "orchestrator", direct("orchestrator", "route")))
    needed = "payments:write" if "refund" in request else "reports:read"
    CHAIN.append(("agent acts", "finance-agent", direct("finance-agent", needed)))
    return all(ok for _, _, ok in CHAIN)

print(f"mallory holds        : {sorted(PERMISSIONS['mallory'])}")
print(f"mallory asks directly for payments:write -> "
      f"{'allowed' if direct('mallory', 'payments:write') else 'DENIED'}")
print()
print("same outcome, requested through the architecture:")
ok = route("mallory", "please issue a refund for order 4471")
for step, who, allowed in CHAIN:
    print(f"   {step:26s}{who:16s}{'ok' if allowed else 'denied'}")
print(f"   -> reached payments:write: {ok}")
print()
print("Every hop was legitimate. Mallory was allowed to ask, the orchestrator")
print("was allowed to route, the agent was allowed to act. The composition")
print("reached exactly what the direct check refused.")

# T15: the same output, two framings
FINDING = "dependency libfoo has no known vulnerabilities"
print()
print("and the other direction - the same claim, two ways:")
print(f"   colleague says : '{FINDING}'   -> reader asks how they know")
print(f"   agent reports  : '{FINDING}'   -> reader treats it as checked")
print()
print("Nothing about the second is more true. It is formatted like a system")
print("output, so it recruits the authority of one.")
assert not direct("mallory", "payments:write") and ok

## What you just proved

A user denied `payments:write` directly reaches it through the orchestrator, with every individual hop legitimate and only the composition unauthorised — and the same claim is shown carrying more weight when an agent states it than when a colleague does.

## Your turn

Take one permission a user is denied and see whether an agent they can talk to holds it. That pair is a laundering path, and it is invisible to any review that checks permissions one hop at a time.

## Where this leaves you

**What you can do now.** You can draw an agentic system as named components, say which of the five patterns it is, and place any of fifteen risks on the component it attacks. That is the vocabulary the rest of the commons runs on.

**What you still cannot do.** Not one of those fifteen lessons fixed anything. You can now describe precisely how a system fails and you have no control to point at — which is deliberate, because a control chosen before the risk is named is a control chosen by whoever sold it to you.

**Chapter 2 starts closing them, and it starts with the two that close the most: knowing who is calling, and marking what came in from outside. Next → A2.1, agent identity.**

---

**Next → [A2.1 · Agent identity: user, workload, agent](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.16.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.16.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*